# 01 · Modelo de obligaciones al retiro

Estima el capital que necesita el inversionista ficticio al cumplir 62 años y el crecimiento anual compuesto requerido desde USD 75.000 sin aportes. [Research Brief V1](../research_brief.md)

El primer retiro ocurre un mes después de cumplir 62 años. Se modelan 360 retiros de USD 5.000 mensuales en poder adquisitivo actual. Cada flujo nominal incorpora inflación desde hoy hasta el pago y se descuenta a la fecha de jubilación. Las tasas son efectivas anuales y constantes. La inflación de 2,5 % y el descuento nominal de 4 % son supuestos ilustrativos editables, pendientes de calibración histórica.

In [ ]:
from math import fsum, isclose, isfinite


def retirement_liability(monthly_real, years_to_retire, retirement_months,
                         initial_assets, inflation, nominal_discount):
    if any(not isfinite(x) or x <= 0 for x in
           (monthly_real, years_to_retire, initial_assets)):
        raise ValueError("Gasto, horizonte y capital inicial deben ser positivos.")
    if type(retirement_months) is not int or retirement_months < 1:
        raise ValueError("La duración del retiro debe ser un entero positivo de meses.")
    if any(not isfinite(x) or x <= -1 for x in (inflation, nominal_discount)):
        raise ValueError("Las tasas deben ser finitas y superiores a -100 %.")

    price_index = (1 + inflation) ** years_to_retire
    cash_flows = []
    for month in range(1, retirement_months + 1):
        years = month / 12
        payment = monthly_real * price_index * (1 + inflation) ** years
        present_value = payment / (1 + nominal_discount) ** years
        cash_flows.append((month, payment, present_value))

    liability = fsum(present_value for _, _, present_value in cash_flows)
    nominal_growth = (liability / initial_assets) ** (1 / years_to_retire)
    return {
        "cash_flows": cash_flows,
        "liability_nominal": liability,
        "liability_today_usd": liability / price_index,
        "required_nominal_cagr": nominal_growth - 1,
        "required_real_cagr": nominal_growth / (1 + inflation) - 1,
    }

In [ ]:
inputs = {
    "monthly_real": 5_000,
    "years_to_retire": 30,
    "retirement_months": 360,
    "initial_assets": 75_000,
}
flat = retirement_liability(**inputs, inflation=0, nominal_discount=0)
matched = retirement_liability(**inputs, inflation=0.025, nominal_discount=0.025)
base = retirement_liability(**inputs, inflation=0.025, nominal_discount=0.04)
higher_discount = retirement_liability(**inputs, inflation=0.025, nominal_discount=0.05)

assert len(base["cash_flows"]) == 360
assert base["cash_flows"][0][0] == 1 and base["cash_flows"][-1][0] == 360
assert isclose(flat["liability_nominal"], 5_000 * 360)
assert isclose(matched["liability_today_usd"], 5_000 * 360)
assert higher_discount["liability_nominal"] < base["liability_nominal"]
assert isclose(75_000 * (1 + base["required_nominal_cagr"]) ** 30,
               base["liability_nominal"])
print("Comprobaciones superadas.")

In [ ]:
scenarios = [
    ("Base", 0.025, 0.04),
    ("Mayor inflación", 0.035, 0.04),
    ("Mayor descuento", 0.025, 0.05),
]
print(f"{'Escenario':<20} {'USD a los 62':>15} {'USD de hoy':>15}"
      f" {'CAGR nominal':>14} {'CAGR real':>12}")
for name, inflation, discount in scenarios:
    result = retirement_liability(**inputs, inflation=inflation, nominal_discount=discount)
    print(f"{name:<20} {result['liability_nominal']:>15,.2f}"
          f" {result['liability_today_usd']:>15,.2f}"
          f" {result['required_nominal_cagr']:>14.2%}"
          f" {result['required_real_cagr']:>12.2%}")

## Interpretación y limitaciones

La obligación nominal a los 62 años será el umbral para evaluar el patrimonio al retiro. Los USD de hoy expresan el poder adquisitivo actual de ese umbral mediante deflación. El descuento financiero de los flujos se aplica hasta los 62 años.

El CAGR requerido es un umbral determinista. La probabilidad de alcanzarlo requiere modelar trayectorias de retorno. El calendario y las tasas son fijos. Impuestos, comisiones, longevidad incierta y riesgo de secuencia quedan pendientes.

**Próximo paso.** Documentar fuentes de inflación y tasas en [02_data_collection.ipynb](02_data_collection.ipynb).